# Bank Loans — a quantitative teardown 🔬
### Real total-return tapes · rate-beta vs credit-beta · HAC + block-bootstrap · crash co-movement

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Free lunch?: Busted](https://img.shields.io/badge/Free_lunch%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We decompose what BKLN *actually is*: a near-zero-**duration** instrument (the rate protection is real) that loads on **equity/credit** (the catch), via its beta to long Treasuries vs to stocks, in the body and in the tail.

> ⚠️ **Not investment advice.** BKLN/TLT/IEF/SPY daily, **total-return** adjusted (`bank_loans.data`, yfinance `auto_adjust=True`); betas via OLS with a HAC *t* and a circular block bootstrap on the difference. BKLN inception bounds the window at 2011-03-03. Sources in [`docs/references.md`](../docs/references.md), reproducible run in [`docs/results.md`](../docs/results.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # study root (bank_loans/)
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from bank_loans import data, strategy

AS_OF = "2026-05-31"
# Try the REAL total-return tapes; fall back to the offline synthetic control if the
# network is blocked. Every cell prints which TAPE it is showing.
try:
    prices = data.load_real(("BKLN", "TLT", "IEF", "SPY")).loc[:AS_OF]
    TAPE = "REAL"
except Exception as e:
    print("network/cache miss -> SYNTHETIC control tape:", type(e).__name__)
    prices, _truth = data.synthetic_four_asset(dur=0.05, credit_beta=0.65, seed=340)
    TAPE = "SYNTHETIC"

BKLN, TLT, IEF, SPY = "BKLN", "TLT", "IEF", "SPY"
rets = strategy.to_returns(prices)
BANNER = ("REAL total-return tape (BKLN/TLT/IEF/SPY, 2011-2026)" if TAPE == "REAL"
          else "SYNTHETIC control tape (dur=0.05, credit_beta=0.65) -- NOT market data")
print(f"[{TAPE}] {BANNER}")
print(f"panel: {len(prices):,} rows  {prices.index[0].date()} -> {prices.index[-1].date()}  fingerprint={data.fingerprint(prices)}")
for c in (BKLN, TLT, IEF, SPY):
    s = strategy.stats(rets[c])
    print(f"  {c}: CAGR {s['cagr']*100:6.2f}%  vol {s['vol']*100:5.1f}%  Sharpe {s['sharpe']:.3f}  maxDD {s['max_dd']*100:6.1f}%")


[REAL] REAL total-return tape (BKLN/TLT/IEF/SPY, 2011-2026)
panel: 3,833 rows  2011-03-03 -> 2026-05-29  fingerprint=a4d0ad1fb115
  BKLN: CAGR   3.73%  vol   5.8%  Sharpe 0.660  maxDD  -24.2%
  TLT: CAGR   2.51%  vol  14.9%  Sharpe 0.241  maxDD  -48.4%
  IEF: CAGR   2.37%  vol   6.5%  Sharpe 0.393  maxDD  -23.9%
  SPY: CAGR  14.10%  vol  17.1%  Sharpe 0.856  maxDD  -33.7%


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| **Signal** | `REAL` | Beta of BKLN to long Treasuries **-0.055** (HAC *t* **-2.95**) — rate risk ≈ 0; it gained **+13.9%** through the 2020–2023 TLT rout. The rate-protection claim clears the *t*≥2 bar. |
| **Tradability** | `FRAGILE` | Real but modest (CAGR **3.73%**, vol **5.8%**); fell in 7/7 equity crashes, gapped **-23.8%** in 2020. Credit + liquidity risk, thin in stress. |
| **Free lunch?** | `BUSTED` | Risk moved from duration to credit: beta to SPY **+0.202** (HAC *t* **+4.48**); equity-beta − rate-beta = **+0.257**, CI **[0.175, 0.349]**. |

> 💡 **In plain words:** BKLN does dodge rate risk — that part is true and certified. But it is a high-yield *credit* asset, not a safe bond; the risk didn't vanish, it changed costume.

## 1 · The claim, steelmanned

- **H₁ (rate protection):** BKLN's beta to the rate factor (long Treasuries) is ≈ 0 — it does not fall when rates rise.
- **H₂ (low risk):** therefore BKLN is a low-risk, bond-like income sleeve.
- **H₃ (free lunch):** you get the high yield without taking on a *new* risk in exchange for giving up duration.

H₁ is **true** (and certified below). H₂/H₃ are **false** — BKLN loads on equity/credit, and the loading shows up exactly in the tail.

## 2 · So what? — what rides on each answer

If H₂/H₃ held, BKLN would be the dominant fixed-income holding in a hiking cycle. If the risk merely *moved* to credit, then a duration-fleeing investor has re-loaded the same portfolio with a recession-timed bomb — and a 'bond replacement' that fails in a stock crash is mis-filed. This is the same 'is this asset what the brochure says?' question the desk asked of preferreds ([Study 338](../../338-preferred-stocks/)).

## 3 · How we'd know — the protocol

Return/vol/drawdown on the total-return tape · full-sample beta of BKLN to TLT and IEF (the rate factor) and to SPY, each with a **HAC *t*** on the OLS-slope influence series · **downside** beta on the worst 10% of equity days · co-movement in every >5% TLT (rate) drawdown and every >10% SPY (equity) drawdown · a **circular block bootstrap** CI on the beta *difference* (equity − rate).

## 4 · The teardown

Headline stats — low vol is real, but read it next to the crashes:

In [2]:
tbl = pd.DataFrame({
  'CAGR %':  [strategy.stats(rets[c])['cagr']*100 for c in (BKLN, TLT, IEF, SPY)],
  'Vol %':   [strategy.stats(rets[c])['vol']*100 for c in (BKLN, TLT, IEF, SPY)],
  'Sharpe':  [strategy.stats(rets[c])['sharpe'] for c in (BKLN, TLT, IEF, SPY)],
  'MaxDD %': [strategy.stats(rets[c])['max_dd']*100 for c in (BKLN, TLT, IEF, SPY)],
}, index=[BKLN, f'{TLT} (long bonds)', f'{IEF} (int bonds)', f'{SPY} (equity)'])
print(f'[{TAPE}] tape')
tbl.round(2)

[REAL] tape


,CAGR %,Vol %,Sharpe,MaxDD %
BKLN,3.730,5.800,0.660,-24.170
TLT (long bonds),2.510,14.930,0.240,-48.350
IEF (int bonds),2.370,6.510,0.390,-23.920
SPY (equity),14.100,17.120,0.860,-33.720


**Rate sensitivity** — OLS beta of BKLN to the duration legs, each with a HAC *t* on the slope's influence series (mean of the influence series = the OLS slope).

In [3]:
b_tlt, t_tlt = strategy.hac_beta_t(rets[BKLN], rets[TLT])
b_ief, t_ief = strategy.hac_beta_t(rets[BKLN], rets[IEF])
print(f'[{TAPE}]  beta BKLN~{TLT} = {b_tlt:+.3f}  (HAC t {t_tlt:+.2f})')
print(f'[{TAPE}]  beta BKLN~{IEF} = {b_ief:+.3f}  (HAC t {t_ief:+.2f})')
print(f'downside beta to {TLT} (worst 10% TLT days) = {strategy.downside_beta(rets[BKLN], rets[TLT]):.3f}')

[REAL]  beta BKLN~TLT = -0.055  (HAC t -2.95)
[REAL]  beta BKLN~IEF = -0.100  (HAC t -2.62)
downside beta to TLT (worst 10% TLT days) = 0.177


> 💡 **In plain words:** the beta to long Treasuries is **-0.055** with a HAC *t* of **-2.95** — past the *t*=2 bar and *negative*. BKLN's price is essentially immune to the level of rates. The rate-protection claim is real and certified.

**Where the risk hides** — BKLN's beta to *equities*, in the body and in the tail.

In [4]:
b_spy, t_spy = strategy.hac_beta_t(rets[BKLN], rets[SPY])
print(f'[{TAPE}]  beta BKLN~{SPY} = {b_spy:+.3f}  (HAC t {t_spy:+.2f})')
print(f'downside beta to {SPY} (worst 10% SPY days) = {strategy.downside_beta(rets[BKLN], rets[SPY]):.3f}')

[REAL]  beta BKLN~SPY = +0.202  (HAC t +4.48)
downside beta to SPY (worst 10% SPY days) = 0.391


> 💡 **In plain words:** BKLN's beta to *stocks* is **+0.202** (HAC *t* **+4.48**) and the downside beta climbs to **0.39** on the worst equity days. The risk didn't vanish when duration did — it moved to credit, and it *intensifies* in the tail.

**Block-bootstrap on the difference** — is BKLN reliably more of an equity/credit bet than a duration bet? We resample the three series jointly in blocks and report the CI.

In [5]:
boot = strategy.bootstrap_beta_diff(rets[BKLN], rets[TLT], rets[SPY], block=21, n_boot=2000, seed=340)
print(f'[{TAPE}] beta-to-{SPY} minus beta-to-{TLT} = {boot["point"]:+.3f}')
print(f'  95% block-bootstrap CI [{boot["ci95"][0]:+.3f}, {boot["ci95"][1]:+.3f}]')
print(f'  BKLN more equity-like in {boot["frac_equity_wins"]*100:.0f}% of resamples')

[REAL] beta-to-SPY minus beta-to-TLT = +0.257
  95% block-bootstrap CI [+0.175, +0.349]
  BKLN more equity-like in 100% of resamples


> 💡 **In plain words:** the point difference is **+0.257**, the CI **[0.175, 0.349]** excludes zero, and BKLN is the more equity-like asset in **100%** of resamples. The duration-for-credit swap is unambiguous.

**Co-movement, both regimes** — rate selloffs (BKLN survives) and equity crashes (BKLN piles on):

In [6]:
rate_eps = strategy.asset_drawdowns(rets[[TLT, BKLN, IEF, SPY]], TLT, thresh=-0.05)
rate = pd.DataFrame([{'peak':e['peak'].date(),'trough':e['trough'].date(),
  f'{TLT} %':e['driver_loss']*100, f'{BKLN} %':e['others'][BKLN]*100} for e in rate_eps])
crash_eps = strategy.asset_drawdowns(rets[[SPY, BKLN, TLT, IEF]], SPY, thresh=-0.10)
crash = pd.DataFrame([{'peak':e['peak'].date(),'trough':e['trough'].date(),
  f'{SPY} %':e['driver_loss']*100, f'{BKLN} %':e['others'][BKLN]*100, f'{TLT} %':e['others'][TLT]*100} for e in crash_eps])
print(f'[{TAPE}] RATE selloffs (>5% TLT): BKLN rose in {(rate[f"{BKLN} %"]>0).sum()}/{len(rate)}')
print(f'[{TAPE}] EQUITY crashes (>10% SPY): BKLN fell in {(crash[f"{BKLN} %"]<0).sum()}/{len(crash)}')
display(rate.round(1)); display(crash.round(1))

[REAL] RATE selloffs (>5% TLT): BKLN rose in 8/10
[REAL] EQUITY crashes (>10% SPY): BKLN fell in 7/7


,peak,trough,TLT %,BKLN %
0,2011-08-10,2011-08-11,-5.000,0.000
1,2011-10-03,2011-10-27,-10.900,6.000
2,2011-12-19,2012-03-19,-10.400,4.600
3,2012-07-25,2013-08-21,-20.700,5.600
4,2015-01-30,2015-06-26,-15.800,1.100
5,2016-07-08,2016-12-14,-17.900,3.100
6,2019-08-28,2019-11-08,-8.200,0.700
7,2020-03-09,2020-03-18,-15.700,-11.800
8,2020-04-21,2020-06-05,-8.600,5.100
9,2020-08-04,2023-10-19,-48.400,13.900


,peak,trough,SPY %,BKLN %,TLT %
0,2011-04-29,2011-10-03,-18.600,-8.400,34.600
1,2015-07-20,2016-02-11,-13.000,-6.400,14.800
2,2018-01-26,2018-02-08,-10.100,-0.600,-3.800
3,2018-09-20,2018-12-24,-19.300,-5.300,4.500
4,2020-02-19,2020-03-23,-33.700,-23.800,14.200
5,2022-01-03,2022-10-12,-24.500,-4.800,-29.300
6,2025-02-19,2025-04-08,-18.800,-3.400,0.800


## 5 · The verdict

Signal `REAL` (beta to long bonds -0.055, HAC *t* -2.95; +13.9% through the 2020–2023 TLT rout). Tradability `FRAGILE` (real but modest: CAGR 3.73%, vol 5.8%; fell 7/7 crashes, −24% gap in 2020). Free-lunch? `BUSTED` (beta to equity +0.202, HAC *t* +4.48; equity−rate beta diff +0.257, CI [0.175, 0.349]).

## 6 · Could you trade it?

Liquidity and capacity are fine in calm markets — BKLN trades heavily. The reservations are structural: (1) the underlying **leveraged loans** are sub-IG credit settling in *weeks*, so the ETF sits on an illiquid market and can gap to a discount in a shock (2020); (2) the fat coupon is compensation for **default + spread + liquidity** risk, not a free premium for shedding duration; (3) the credit-tail beta means it correlates with your stocks exactly when you'd want a diversifier. As a tactical duration hedge: defensible. As a *safe* bond substitute: a mirage of the kind this desk keeps finding.

## 7 · Going further

- Run the **two-factor** (duration + credit) regression jointly and read off the loadings with HAC errors — the single-factor cut here already tells the story, but the joint fit quantifies the swap.
- Decompose BKLN total return into **coupon vs price** to see how much of the 3.7%/yr is carry being clawed back by 2020/2022 capital losses.
- Add **HYG** (high-yield bonds) and **SRLN** (active loans): is the credit-tail behaviour structural to leveraged credit, or fund-specific?